# Семинар 4. Pandas: Series, DataFrame, загрузка и просмотр данных. Старт сквозного кейса

- Переименуйте файл в формате `Группа-Фамилия-Имя-seminar-04.ipynb`, например `2MP9-Ivanov-Ivan-seminar-04.ipynb`.
- Ноутбук читает файл `datasets/legal/dtp-bryansk.csv` из репозитория курса: путь в ячейке задан от папки `notebooks/`, поэтому папки `notebooks/` и `datasets/` должны лежать рядом, как в репозитории. Описание данных — в `datasets/legal/README.md`.
- Упражнения к занятию лежат в папке `exercises/`: листок `seminar-04-tasks.md` с формулировками и ноутбук `seminar-04-tasks.ipynb` с заготовками и тестами; они решаются на тех же данных, но по Московской области. Ответы вводятся в тест Moodle «Семинар 4».
- Шпаргалка по функциям: `reference/pandas.md`.

## 1. Зачем pandas после NumPy

На семинарах 2 и 3 картотека дел жила в трёх массивах: номера, цены, сроки. Массивы приходилось держать согласованными по позиции: перемешал один — перемешивай все, отобрал по маске цены — не забудь отобрать номера. Настоящие данные приходят таблицей: у столбцов есть имена и разные типы, числа соседствуют с текстом и датами, а таблица читается из файла одной командой. Для таблиц в Python есть библиотека **pandas**; внутри её столбцов лежат массивы NumPy, поэтому всё, что умеет массив, умеет и столбец.

Сквозной кейс курса: дорожно-транспортные происшествия с пострадавшими. Каждое такое ДТП порождает правовые последствия: административную ответственность по ст. 12.24 КоАП РФ или уголовную по ст. 264 УК РФ, страховые споры по ОСАГО, иски к дорожным службам за недостатки дорог. ГИБДД публикует данные о каждом ДТП с пострадавшими, проект «Карта ДТП» собирает их по регионам. Сегодня Брянская область за 2015–2025 годы: 10 114 ДТП, по одной строке на происшествие. Загрузка занимает одну строку.

In [ ]:
# Импорт библиотеки pandas
import pandas as pd

In [ ]:

# Загрузка данных о ДТП в Брянской области из CSV-файла
dtp = pd.read_csv('../datasets/legal/dtp-bryansk.csv', index_col='id')   # ДТП Брянской области
dtp.head()                                                                # первые пять ДТП

Каждая строка — одно ДТП: дата и время, район, вид, тяжесть, освещение, погода, недостатки дороги, число участников, машин, погибших и раненых, координаты. Слева стоит номер ДТП с портала: при чтении файла он сделан индексом таблицы, и дальше строки будут находиться по этому номеру, как дела по номеру в картотеке. Что означает каждый столбец, записано в `datasets/legal/README.md`; в упражнениях та же структура, но по Московской области.

## 2. Series: столбец с метками

Основные типы pandas: **Series** — один столбец, **DataFrame** — таблица из столбцов. У Series, в отличие от массива, кроме значений есть **индекс**: метка у каждого значения. Картотека из двенадцати дел прошлых семинаров становится одним Series: цены исков со значениями и номера дел в индексе. Массив `numbers` больше не нужен: номер едет вместе с ценой.

In [ ]:
amounts = pd.Series([48_000, 250_000, 1_200_000, 75_000, 320_000, 15_000,
                     640_000, 98_000, 410_000, 135_000, 27_000, 560_000],
                    index=range(1, 13), name='amount')                   # цены исков по номерам дел
amounts

In [ ]:
amounts.values, amounts.index                                            # значения и метки

`values` — обычный массив NumPy, `index` — метки. Арифметика и статистики работают как у массива: операция с числом применяется к каждому значению, индекс сохраняется. Методы среднего, медианы и максимума есть у самого Series, медиана здесь метод, а не отдельная функция.

In [ ]:
amounts / 1000                                                           # цены в тысячах рублей

In [ ]:
amounts.mean(), amounts.median(), amounts.max()                          # статистики цен

Поиск по столбцу возвращает не позицию, а метку: `idxmax` даёт номер самого дорогого дела, без второго массива. Маска отбирает значения вместе с метками: номера дел дороже 100 000 видны сразу.

In [ ]:
amounts.idxmax()                                                         # номер самого дорогого дела

In [ ]:
amounts[amounts > 100_000]                                               # дела дороже 100 000

Два способа обратиться к элементу: `loc` по метке и `iloc` по позиции. Метка 3 — дело номер 3, позиция 3 — четвёртое дело по счёту, потому что позиции считаются с нуля. Дальше эта пара будет главным инструментом выбора строк в таблице.

In [ ]:
amounts.loc[3], amounts.iloc[3]                                          # дело 3 и четвёртое по счёту

## 3. DataFrame и загрузка файла

Таблица `dtp` загружена в разделе 1 одной строкой; повторим чтение, выписав параметры, которые обычно приходится менять. `read_csv` читает текстовый файл с разделителями: первый аргумент — путь к файлу от папки, в которой лежит ноутбук; `index_col='id'` делает столбец с номером ДТП индексом; `sep` — разделитель значений, `encoding` — кодировка файла. Для файла курса это значения по умолчанию, запятая и UTF-8; у файлов с российских порталов часто `sep=';'` и `encoding='cp1251'`, иначе русские буквы читаются как мусор. Сначала таблицу разглядывают: форма, столбцы, типы.

In [ ]:
# Загрузка данных о ДТП в Брянской области из CSV-файла c явным указанием параметров
dtp = pd.read_csv('../datasets/legal/dtp-bryansk.csv', index_col='id',
                  sep=',', encoding='utf-8')                             # те же данные, параметры явно
dtp.shape                                                                # число ДТП и столбцов

In [ ]:
dtp.columns                                                              # названия столбцов

In [ ]:
dtp.dtypes                                                               # типы столбцов

Типы у столбцов разные: `int64` — целые, `float64` — дробные, `str` — текст. В одном столбце тип один, как в массиве, но в таблице столбцы разных типов живут рядом. `info` показывает всё сразу: сколько строк, сколько непустых значений в каждом столбце, типы и объём памяти.

In [ ]:
dtp.info()                                                               # строки, пропуски, типы

В столбце `road_defect` непустых значений 3 299 из 10 114: у остальных ДТП недостатки дорожных условий не установлены, и в файле там пусто. Так в таблице выглядят пропуски; работать с ними будем на семинаре 6. Заглянуть в конец таблицы и в случайные строки помогают `tail` и `sample`; `random_state` делает случайный выбор одинаковым у всех.

In [ ]:
dtp.tail(3)                                                              # последние три ДТП

In [ ]:
dtp.sample(3, random_state=4)                                            # три случайных ДТП

## 4. Первый взгляд на данные

`describe` считает по каждому числовому столбцу число значений, среднее, стандартное отклонение, минимум, квартили и максимум — то, что на семинаре 3 собиралось по одной статистике. Координаты тоже числа, и для них сводка бессмысленна; это не ошибка, просто строки о `lat` и `lon` пропускаются глазами. Сводку одного столбца даёт `describe` у Series.

In [ ]:
dtp.describe().round(2)                                                  # сводка числовых столбцов

In [ ]:
dtp['participants_count'].describe()                                     # участники в одном ДТП

Для текстовых столбцов главный инструмент — `value_counts`: сколько раз встречается каждое значение, по убыванию. С `normalize=True` вместо чисел доли. `nunique` считает разные значения, `unique` перечисляет их.

In [ ]:
dtp['category'].value_counts()                                           # ДТП по видам

In [ ]:
dtp['severity'].value_counts(normalize=True).round(3)                    # доли по тяжести

In [ ]:
dtp['light'].nunique(), dtp['light'].unique()                            # условия освещения

Значения категорий берутся точно как в таблице, со всеми пробелами и буквами: «темное» в столбце `light` написано без буквы ё, как в источнике. Удобнее всего копировать значение из вывода `value_counts`. Среднее у текстового столбца не считается: следующая ячейка намеренно завершается ошибкой.

In [ ]:
dtp['category'].mean()                                                   # среднее по тексту: ошибка

## 5. Столбцы и строки

Столбец берётся по имени в квадратных скобках и оказывается Series с индексом таблицы. Несколько столбцов — список имён во вторых квадратных скобках, результат снова таблица.

In [ ]:
dtp['dead_count']                                                        # столбец погибших: Series

In [ ]:
dtp[['datetime', 'district', 'dead_count']].head()                       # три столбца: таблица

Два имени через запятую без внутренних скобок pandas принимает за одно составное имя и не находит его: ошибка `KeyError` в следующей ячейке намеренная.

In [ ]:
dtp['datetime', 'district']                                              # без внутренних скобок: ошибка

Строки выбираются так же, как элементы Series: `loc` по метке, здесь по номеру ДТП, `iloc` по позиции. Строка таблицы — тоже Series, где индексом стали названия столбцов. Через запятую после метки указывается столбец: одно поле карточки ДТП. Позиция 2 — третье ДТП по счёту, его номер 177; ДТП с номером 2 в таблице нет, и `dtp.loc[2]` дал бы `KeyError`.

In [ ]:
dtp.loc[31]                                                              # ДТП номер 31

In [ ]:
dtp.loc[31, 'district']                                                  # район этого ДТП

In [ ]:
dtp.iloc[2]                                                              # третье ДТП по счёту

In [ ]:
dtp.loc[[31, 133], ['datetime', 'category', 'dead_count']]               # два ДТП, три поля

Срезы: `iloc` работает как срез массива, правая граница не входит; `loc` берёт метки от и до включительно, потому что метка — это имя, а не номер по счёту. На Series с ценами разница видна: `loc[3:5]` даёт три дела, `iloc[3:5]` — два.

In [ ]:
dtp.iloc[:3]                                                             # первые три ДТП

In [ ]:
amounts.loc[3:5], amounts.iloc[3:5]                                      # метки включительно, позиции нет

## 6. Маски

Сравнение столбца с числом или строкой даёт логический Series — маску, такую же, как в NumPy. Сумма маски — число подходящих ДТП, среднее — их доля. Маска в квадратных скобках таблицы оставляет только подходящие строки.

In [ ]:
with_dead = dtp['dead_count'] > 0                                        # маска: ДТП с погибшими
with_dead.sum(), with_dead.mean().round(3)                               # число и доля

In [ ]:
dtp[with_dead].head()                                                    # сами ДТП с погибшими

Условия соединяются `&`, `|` и `~`, каждое в скобках; `isin` проверяет вхождение в список значений, `between` — попадание в диапазон с включёнными границами. Ночные ДТП — три значения освещения сразу.

In [ ]:
night = dtp['light'].isin(['В темное время суток, освещение включено',
                           'В темное время суток, освещение отсутствует',
                           'В темное время суток, освещение не включено'])   # маска: ночные ДТП
night.sum(), night.mean().round(3)                                            # число и доля ночных

In [ ]:
pedestrians_night = (dtp['category'] == 'Наезд на пешехода') & night     # пешеходы ночью
pedestrians_night.sum()

In [ ]:
dtp['injured_count'].between(3, 5).sum()                                 # ДТП с 3–5 ранеными

Слово `and` вместо `&` даёт ту же ошибку, что в NumPy: у маски из тысяч значений нет одного «истинно» или «ложно». Ячейка ниже завершается ошибкой намеренно.

In [ ]:
(dtp['category'] == 'Наезд на пешехода') and night                       # and вместо &: ошибка

Маска и список столбцов вместе в `loc`: строки по условию, поля по выбору. Так составляется выборка для отчёта.

In [ ]:
dtp.loc[dtp['dead_count'] >= 3, ['datetime', 'district', 'category', 'dead_count']]   # ДТП с тремя и более погибшими

## 7. Мини-анализ и сохранение

Несколько вопросов к данным, на которые таблица отвечает без группировок и сортировок: только столбцы, маски и статистики. Доля наездов на пешеходов среди всех ДТП; доля ДТП с погибшими днём и ночью; самое тяжёлое ДТП области.

In [ ]:
(dtp['category'] == 'Наезд на пешехода').mean().round(3)                 # доля наездов на пешеходов

In [ ]:
day = dtp['light'] == 'Светлое время суток'                              # маска: дневные ДТП
(dtp.loc[day, 'dead_count'] > 0).mean().round(3), (dtp.loc[night, 'dead_count'] > 0).mean().round(3)   # доля ДТП с погибшими днём и ночью

In [ ]:
worst = dtp['dead_count'].idxmax()                                       # номер самого тяжёлого ДТП
dtp.loc[worst]

Ночью доля ДТП с погибшими заметно выше дневной; на семинарах 11–13 такие различия будут проверяться на случайность. Выборка для отчёта сохраняется в файл методом `to_csv`: файл появится рядом с ноутбуком, индекс с номерами ДТП записывается первым столбцом.

In [ ]:
severe = dtp[dtp['dead_count'] >= 3]                                     # ДТП с тремя и более погибшими
severe.to_csv('dtp-bryansk-severe.csv')                                  # выборка в файл
len(severe)

## Итоги

Таблица читается одной командой `read_csv`, номер ДТП становится индексом; `shape`, `dtypes` и `info` показывают устройство таблицы, `describe` и `value_counts` — содержимое столбцов; столбец — это Series с метками, строка выбирается через `loc` по метке или `iloc` по позиции; маски отбирают строки так же, как в NumPy, а `isin` и `between` заменяют цепочки условий. Переходите к упражнениям: откройте `exercises/seminar-04-tasks.ipynb`, формулировки в `exercises/seminar-04-tasks.md`; данные там по Московской области за 2022–2025 годы.